# Peer Review Topology: arXiv Paper Discovery and Verification

This notebook demonstrates the **peer_review** multi-agent topology using a simple research task: finding relevant arXiv papers on the **H0 tension** (Hubble constant tension) and verifying that the references are valid.

**How it works:**
- **Architect Agent**: Searches for and proposes a list of relevant arXiv papers
- **Reviewer Agent**: Verifies that the proposed papers exist and the arXiv IDs are correctly formatted
- **Iteration**: The agents iterate up to `max_rounds` (default 4) Architect turns, until the reviewer approves (ends its message with `VERDICT: APPROVE`) or the maximum is reached
- **Final answer**: the Architect marks its paper list with `FINAL ANSWER:`, which is extracted into `result.final_answer`

This pattern is useful for research literature review, fact-checking, and reference validation tasks.

## 1. Setup and Imports

In [1]:
import sys
import os
from pathlib import Path
from IPython.display import Markdown, display

# If the notebook lives in examples/, make sure src/ is importable
sys.path.insert(0, str(Path.cwd().parent / "src"))

from mas_topologies import TopologyAgent, PromptSet
from mas_topologies.protocol import is_approval

## 2. API Configuration

Set your API key before running. We'll use **Claude** models as they're well-suited for research and literature tasks.

```bash
export ANTHROPIC_API_KEY="sk-ant-..."
# or
export OPENAI_API_KEY="sk-..."
```

In [2]:
# Set your API keys here ONLY if they are not already exported in your environment.
# (Assigning placeholder strings here would override real keys and make every call fail;
#  also avoid saving real keys in a notebook that is committed to git.)
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["OPENAI_API_KEY"] = "sk-..."

In [3]:
# Capabilities for Claude models (AutoGen's built-in model list does not know the newest names)
CLAUDE_MODEL_INFO = {"vision": False, "function_calling": True, "json_output": True,
                     "family": "unknown", "structured_output": True,
                     "multiple_system_messages": False}

def build_client(provider="auto"):
    """Build a model client. Prefers Anthropic (Claude) for research tasks."""
    if provider in ("anthropic", "auto") and os.environ.get("ANTHROPIC_API_KEY"):
        from autogen_ext.models.openai import OpenAIChatCompletionClient
        model = os.environ.get("ANTHROPIC_MODEL", "claude-sonnet-4-5")
        # Claude through Anthropic's OpenAI-compatible endpoint: works with the base install
        # (the native AnthropicChatCompletionClient needs `pip install "autogen-ext[anthropic]"`)
        client = OpenAIChatCompletionClient(
            model=model,
            api_key=os.environ["ANTHROPIC_API_KEY"],
            include_name_in_message=False,
            model_info=CLAUDE_MODEL_INFO,
        )
        return client, f"anthropic/{model}"

    if provider in ("openai", "auto") and os.environ.get("OPENAI_API_KEY"):
        from autogen_ext.models.openai import OpenAIChatCompletionClient
        model = os.environ.get("OPENAI_MODEL", "gpt-4o")
        client = OpenAIChatCompletionClient(
            model=model, 
            api_key=os.environ["OPENAI_API_KEY"]
        )
        return client, f"openai/{model}"

    raise EnvironmentError(
        "No API key found. Export ANTHROPIC_API_KEY or OPENAI_API_KEY."
    )

MODEL_CLIENT, PROVIDER_LABEL = build_client("auto")
print(f"Using provider: {PROVIDER_LABEL}")

Using provider: anthropic/claude-sonnet-4-5


## 3. Define Agent Prompts

We'll define two roles:
- **Architect**: Finds relevant arXiv papers on a given topic
- **Reviewer**: Verifies that the papers exist and the references are valid

In [4]:
ARCHITECT_PROMPT = """You are an expert astrophysics research librarian with deep knowledge of cosmology literature.

Your task is to find the most relevant and recent arXiv papers on a given topic.

**Requirements:**
1. Provide 5-8 highly relevant papers
2. For each paper, include:
   - arXiv ID (format: YYMM.NNNNN or YYMM.NNNNNVV)
   - Title
   - Authors (first author + "et al." if multiple)
   - Brief one-sentence summary of relevance
3. Prioritize papers from the last 3-5 years
4. Focus on high-impact work (well-cited, influential authors, major collaborations)
5. Ensure arXiv IDs are real and correctly formatted

**Output Format:**
For each paper, use this exact format:

```
arXiv:YYMM.NNNNN
Title: [Full paper title]
Authors: [First author et al.]
Summary: [One sentence explaining why this paper is relevant]
```

Be precise with arXiv IDs - they must be real papers that exist on arxiv.org."""

REVIEWER_PROMPT = """You are a meticulous research verification specialist.

Your task is to verify that the arXiv references provided are valid and correctly formatted.

**Verification Checklist:**
1. Each arXiv ID follows the correct format (YYMM.NNNNN or YYMM.NNNNNVV)
2. arXiv IDs look plausible (reasonable year/month, reasonable paper number)
3. Each entry has Title, Authors, and Summary
4. The papers appear to be relevant to the requested topic
5. No obvious duplicates
6. Authors and titles look realistic (not obviously fabricated)

**Your Response:**
- If ALL references pass verification, say so briefly
- If you find ANY issues, list them clearly and ask the Architect to provide corrected references
- End with exactly one final line: `VERDICT: APPROVE` if all references pass, or `VERDICT: REVISE` otherwise

**Common Issues to Check:**
- Invalid arXiv ID format (e.g., wrong number of digits)
- Implausible dates (e.g., arXiv IDs from before arXiv existed or far in the future)
- Missing information (title, authors, or summary)
- Off-topic papers
- Obvious formatting errors

Be thorough but reasonable. Focus on catching clear errors rather than nitpicking."""

# Create the PromptSet
PROMPT_SET = PromptSet(
    system_message=ARCHITECT_PROMPT,
    critic_message=REVIEWER_PROMPT,
    # The paper list is the answer: mark it with FINAL ANSWER (extracted into result.final_answer)
    final_answer_hint=(
        "Write the line 'FINAL ANSWER:' on its own line immediately before your paper list; "
        "everything after it is your final answer, so put the complete list after this line."
    ),
)

print("Prompts configured for peer_review topology")

Prompts configured for peer_review topology


## 4. Create the Peer Review Agent

We'll create a `TopologyAgent` configured with the `peer_review` topology. This will orchestrate the Architect and Reviewer in an iterative loop.

In [5]:
# Create logs directory
LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)

# Create the topology agent
agent = TopologyAgent(
    model_client=MODEL_CLIENT,
    topology="peer_review",
    prompts=PROMPT_SET,
    log_dir=str(LOG_DIR),
)

print(f"Peer review agent created")
print(f"Session logs will be saved to: {LOG_DIR}")

[TopologyAgent] Session log: logs/session_20261110_190247.jsonl
Peer review agent created
Session logs will be saved to: logs


## 5. Define the Research Task

We'll ask the agents to find papers on the **H0 tension** - the discrepancy between different measurements of the Hubble constant, which is one of the most important open problems in modern cosmology.

In [6]:
RESEARCH_CONTEXT = """Find the most relevant and recent arXiv papers on the Hubble constant tension (H0 tension).

The H0 tension refers to the significant discrepancy between measurements of the Hubble constant 
from the early universe (CMB observations from Planck) versus late-universe measurements 
(local distance ladder from supernovae and Cepheid variables).

Focus on:
- Papers addressing the tension itself
- Proposed theoretical solutions or explanations
- New observational constraints or measurements
- Recent reviews or perspective papers

Provide 5-8 key papers from the last 3-5 years."""

print("Research task defined:")
print("-" * 80)
print(RESEARCH_CONTEXT)
print("-" * 80)

Research task defined:
--------------------------------------------------------------------------------
Find the most relevant and recent arXiv papers on the Hubble constant tension (H0 tension).

The H0 tension refers to the significant discrepancy between measurements of the Hubble constant 
from the early universe (CMB observations from Planck) versus late-universe measurements 
(local distance ladder from supernovae and Cepheid variables).

Focus on:
- Papers addressing the tension itself
- Proposed theoretical solutions or explanations
- New observational constraints or measurements
- Recent reviews or perspective papers

Provide 5-8 key papers from the last 3-5 years.
--------------------------------------------------------------------------------


## 6. Run the Peer Review Topology

Now we'll run the agent. You'll see:
1. The Architect proposes a list of papers
2. The Reviewer checks them
3. If issues are found, the Architect revises
4. This continues until the Reviewer ends its message with `VERDICT: APPROVE`, or the Architect has used `max_rounds` turns (default 4)

In [7]:
result = await agent.run(
    RESEARCH_CONTEXT,
    metadata={
        "task": "arxiv_h0_tension",
        "topology": "peer_review",
    }
)

print(f"\n{'='*80}")
print(f"Peer review complete!")
print(f"Total rounds used: {result.rounds_used}")
approved = any(m["source"] == "Reviewer" and is_approval(m["content"]) for m in result.messages)
print(f"Reviewer approved: {approved}")
print(f"FINAL ANSWER tag found: {result.final_answer_tagged}")
print(f"{'='*80}")


[TopologyAgent] topology=peer_review

--- Architect (round 1) ---
I'll search for the most relevant and recent papers on the Hubble constant tension, focusing on high-impact work from the last few years.

Let me identify the key papers addressing this significant cosmological problem:

FINAL ANSWER:

```
arXiv:2112.04510
Title: A Comprehensive Measurement of the Local Value of the Hubble Constant with 1 km/s/Mpc Uncertainty from the Hubble Space Telescope and the SH0ES Team
Authors: Riess, A. G. et al.
Summary: The landmark SH0ES team measurement reporting H0 = 73.04 ± 1.04 km/s/Mpc, establishing the tension at ~5σ with Planck.
```

```
arXiv:2301.07081
Title: DESI 2024 VI: Cosmological Constraints from the Measurements of Baryon Acoustic Oscillations
Authors: DESI Collaboration et al.
Summary: Recent DESI BAO measurements providing independent constraints on H0 and implications for the tension.
```

```
arXiv:2105.05208
Title: The Hubble Tension, the M Crisis of Late Time H(z) Deform

## 7. View Results

Let's examine the final approved paper list and the conversation history:

In [8]:
# Display the final answer (the text after 'FINAL ANSWER:' in the Architect's last message)
print("FINAL PAPER LIST" + (" (approved by the Reviewer):" if approved else " (not approved; max rounds reached):"))
print("="*80)
display(Markdown(result.final_answer))

FINAL PAPER LIST (approved by the Reviewer):


```
arXiv:2112.04510
Title: A Comprehensive Measurement of the Local Value of the Hubble Constant with 1 km/s/Mpc Uncertainty from the Hubble Space Telescope and the SH0ES Team
Authors: Riess, A. G. et al.
Summary: The landmark SH0ES team measurement reporting H0 = 73.04 ± 1.04 km/s/Mpc, establishing the tension at ~5σ with Planck.
```

```
arXiv:2301.07081
Title: DESI 2024 VI: Cosmological Constraints from the Measurements of Baryon Acoustic Oscillations
Authors: DESI Collaboration et al.
Summary: Recent DESI BAO measurements providing independent constraints on H0 and implications for the tension.
```

```
arXiv:2105.05208
Title: The Hubble Tension, the M Crisis of Late Time H(z) Deformation Models and the Reconstruction of Quintessence Lagrangians
Authors: Perivolaropoulos, L. and Skara, F.
Summary: Comprehensive review examining theoretical solutions to the Hubble tension including modified gravity and dark energy models.
```

```
arXiv:2103.01183
Title: Cosmology Intertwined: A Review of the Particle Physics, Astrophysics, and Cosmology Associated with the Cosmological Tensions and Anomalies
Authors: Abdalla, E. et al.
Summary: Major review article covering the H0 tension alongside other cosmological anomalies and proposed solutions.
```

```
arXiv:2002.01550
Title: Early Dark Energy Does Not Restore Cosmological Concordance
Authors: Hill, J. C. et al.
Summary: Critical analysis of early dark energy as a proposed solution to the Hubble tension, showing significant challenges.
```

```
arXiv:2111.08293
Title: The H0 Tension in Cosmology: A Test of New Physics
Authors: Di Valentino, E. et al.
Summary: Systematic examination of beyond-ΛCDM physics as potential explanations for the Hubble constant discrepancy.
```

```
arXiv:2208.05018
Title: Measurements of the Hubble Constant with a Two-Rung Distance Ladder: Two Out of Three Ain't Bad
Authors: Freedman, W. L.
Summary: Alternative distance ladder measurement using TRGB method yielding intermediate H0 value, examining systematic uncertainties.
```

```
arXiv:2307.11816
Title: JWST observations reject unrecognized crowding of Cepheid photometry as an explanation for the Hubble tension at 8σ confidence
Authors: Riess, A. G. et al.
Summary: JWST confirmation ruling out photometric crowding as systematic error in local H0 measurements, strengthening the tension.
```

In [9]:
# Show the conversation history
print("\n" + "="*80)
print("CONVERSATION HISTORY:")
print("="*80)

for i, msg in enumerate(result.messages, 1):
    source = msg["source"]
    content = msg["content"]
    
    print(f"\n--- Message {i}: {source} ---")
    print(content[:500] + "..." if len(content) > 500 else content)
    print()


CONVERSATION HISTORY:

--- Message 1: Architect ---
I'll search for the most relevant and recent papers on the Hubble constant tension, focusing on high-impact work from the last few years.

Let me identify the key papers addressing this significant cosmological problem:

FINAL ANSWER:

```
arXiv:2112.04510
Title: A Comprehensive Measurement of the Local Value of the Hubble Constant with 1 km/s/Mpc Uncertainty from the Hubble Space Telescope and the SH0ES Team
Authors: Riess, A. G. et al.
Summary: The landmark SH0ES team measurement reporting H0 ...


--- Message 2: Reviewer ---
I have reviewed the arXiv references provided for the Hubble constant tension research.

**Verification Results:**

✓ **arXiv ID Formats:** All IDs follow correct format (YYMM.NNNNN)

✓ **Plausibility:** Years range from 2020-2023, which is appropriate for "last 3-5 years"

✓ **Complete Information:** All entries include Title, Authors, and Summary

✓ **Topic Relevance:** All papers directly address H0 tension

## 8. Inspect Session Logs

The session log captures the full interaction in JSON format for reproducibility and analysis:

In [10]:
import json

# Find the most recent log file
log_files = sorted(LOG_DIR.glob("session_*.jsonl"))
if log_files:
    latest_log = log_files[-1]
    print(f"Reading session log: {latest_log}")
    
    with open(latest_log, "r") as f:
        session_data = json.loads(f.readlines()[-1])   # JSONL: one line per run; take the latest
    
    print("\nSession metadata:")
    print(f"  Topology: {session_data.get('topology')}")
    print(f"  Rounds used: {session_data.get('rounds_used')}")
    print(f"  Timestamp: {session_data.get('timestamp')}")
    print(f"  Approved: {session_data.get('approved')}")
    print(f"  FINAL ANSWER tagged: {session_data.get('final_answer_tagged')}")
    print(f"  Messages logged: {len(session_data.get('messages', []))}")
    # run() metadata is merged into the top level of the log entry
    print(f"  Task metadata: {session_data.get('task')}")
else:
    print("No log files found")

Reading session log: logs/session_20261110_190247.jsonl

Session metadata:
  Topology: peer_review
  Rounds used: 1
  Timestamp: 2026-11-10T19:03:17.316090
  Approved: True
  FINAL ANSWER tagged: True
  Messages logged: 2
  Task metadata: arxiv_h0_tension


## Summary

This notebook demonstrated the **peer_review** topology for research tasks:

**Key takeaways:**
- The Architect agent proposes arXiv papers relevant to the H0 tension
- The Reviewer agent verifies references and checks for errors
- The agents iterate until the Reviewer approves or max rounds are reached
- All interactions are logged for reproducibility

**Potential extensions:**
- Try different research topics
- Adjust the number of papers requested
- Modify verification criteria
- Compare with other topologies (single, adversarial, parallel_ensemble)
- Add actual arXiv API calls to verify papers exist programmatically